# 04 — Street View Segmentation

`POST /v1/streetview` segments ground-level imagery (buildings, roads, vegetation, sky…) at a chosen viewing angle.

**Plan:** Premium only.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv; load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
client = FortyGuardClient()

In [ ]:
response = client.street_view_segmentation(
    latitude=40.7128,
    longitude=-74.0060,
    vertical_angle=10.0,
    horizontal_angle=90.0,
    back_view=False,
)
result = response['result']
print('Coordinates:', result.get('coordinates'))
print('Front keys :', list(result.get('front', {}).keys()))

In [ ]:
import base64, io
from PIL import Image
import matplotlib.pyplot as plt

def _decode(b64):
    if not b64: return None
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

front = result.get('front', {})
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, title in [(axes[0], 'original_image', 'Street view'), (axes[1], 'segmented_image', 'Segmentation')]:
    img = _decode(front.get(key))
    if img is not None:
        ax.imshow(img)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
segments = front.get('segments', {})
print('Class coverage:')
for cls, pct in sorted(segments.items(), key=lambda kv: kv[1], reverse=True):
    print(f'  {cls:>25}: {pct}')